# 02 — Customer Churn Prediction and Model Comparison

**Goal:** Rank customers by churn risk, compare simple and nonlinear classifiers, evaluate business contact capacity, and use the untouched test partition exactly once. This notebook uses the same row indices and joint treatment/outcome stratification as Notebook 1.

**Primary development metric:** Average Precision (AP), with ROC-AUC, threshold-dependent metrics, calibration, and Precision@K as supporting measures.

**Interpretation:** Churn prediction identifies risk; it does not prove why customers churn or whether contact would prevent churn. Keep `t` separate from the risk predictors.

## 1. Recreate the Notebook 1 Data Split

The code below reproduces the original 60%/20%/20% split with `random_state=42`, stratified by the four treatment–outcome combinations. Using the identical split makes subsequent experiments comparable.

Keep the source file unchanged.

In [ ]:
# ============================================================
# NOTEBOOK 2 SETUP: REPRODUCE NOTEBOOK 1 DATA SPLIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Find the raw CSV from either the project root or notebooks/.
roots = [Path.cwd(), Path.cwd().parent]
filename = "Orange Belgium_churn_uplift_anonymized.csv"
paths = [root / folder / filename for root in roots
         for folder in (Path("data/raw"), Path("."))]
data_path = next((p for p in paths if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Place the original Orange Belgium CSV in data/raw/ or the project root.")

df = pd.read_csv(data_path)
X = df.drop(columns=["y", "t", "FACTOR3"]).copy()
y = df["y"].copy()  # Churn outcome
t = df["t"].copy()  # Treatment assignment

# Preserve all four treatment-outcome combinations when splitting.
strata = t.astype(str) + "_" + y.astype(str)
train_indices, temp_indices = train_test_split(
    df.index.to_numpy(), test_size=0.40, random_state=42, stratify=strata
)
val_indices, test_indices = train_test_split(
    temp_indices, test_size=0.50, random_state=42,
    stratify=strata.loc[temp_indices]
)
X_train, X_val = X.loc[train_indices].copy(), X.loc[val_indices].copy()
y_train, y_val = y.loc[train_indices].copy(), y.loc[val_indices].copy()
t_train, t_val = t.loc[train_indices].copy(), t.loc[val_indices].copy()

# Keep test arrays reserved for a later final assessment.
X_test, y_test, t_test = (X.loc[test_indices].copy(),
                         y.loc[test_indices].copy(),
                         t.loc[test_indices].copy())
assert len(X_train) == 7137 and len(X_val) == 2379 and len(X_test) == 2380
assert X_train.shape[1] == 177
print("Data split recreated; validation is ready, test remains reserved.")

## 2. Majority-Class Baseline

A `DummyClassifier` always predicts the majority class (no churn). Because churn is rare, it has high accuracy but cannot identify customers who churn. It provides a minimum reference point for positive-class recall and F1.

In [ ]:
# ============================================================
# PROJECT 3 - STEP 4A: MAJORITY-CLASS BASELINE MODEL
# ============================================================

# Import the simplest classification model.
from sklearn.dummy import DummyClassifier

# Import classification evaluation metrics.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. CREATE THE BASELINE CLASSIFIER
# ------------------------------------------------------------

# This classifier always predicts the most frequent
# class observed in the training dataset.
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# ------------------------------------------------------------
# 2. TRAIN THE BASELINE MODEL
# ------------------------------------------------------------

# Fit the model using training predictors and churn labels.
# The classifier learns only which outcome is most common.
baseline_model.fit(X_train, y_train)

# ------------------------------------------------------------
# 3. GENERATE VALIDATION PREDICTIONS
# ------------------------------------------------------------

# Predict churn outcomes for validation customers.
# Do not use X_test at this stage.
baseline_predictions = baseline_model.predict(X_val)

# ------------------------------------------------------------
# 4. CALCULATE CLASSIFICATION METRICS
# ------------------------------------------------------------

baseline_accuracy = accuracy_score(
    y_val,
    baseline_predictions
)

baseline_precision = precision_score(
    y_val,
    baseline_predictions,
    zero_division=0
)

baseline_recall = recall_score(
    y_val,
    baseline_predictions,
    zero_division=0
)

baseline_f1 = f1_score(
    y_val,
    baseline_predictions,
    zero_division=0
)

# ------------------------------------------------------------
# 5. DISPLAY MODEL PERFORMANCE
# ------------------------------------------------------------

print("MAJORITY-CLASS BASELINE RESULTS")
print("-" * 55)

print(f"Accuracy: {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall: {baseline_recall:.4f}")
print(f"F1-score: {baseline_f1:.4f}")

# ------------------------------------------------------------
# 6. DISPLAY THE CONFUSION MATRIX
# ------------------------------------------------------------

# Fix the class order:
# 0 = Did not churn
# 1 = Churned
baseline_cm = confusion_matrix(
    y_val,
    baseline_predictions,
    labels=[0, 1]
)

print("\nCONFUSION MATRIX")
print("-" * 55)

print(baseline_cm)

# ------------------------------------------------------------
# 7. DISPLAY THE CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nCLASSIFICATION REPORT")
print("-" * 55)

print(
    classification_report(
        y_val,
        baseline_predictions,
        labels=[0, 1],
        target_names=["No Churn", "Churn"],
        zero_division=0
    )
)

## 3. Logistic Regression: Training Pipeline

`StandardScaler` scales the 160 numerical features. `OneHotEncoder` encodes the 17 nonconstant categorical features. A `Pipeline` fits these preprocessing steps on **training data only**, reducing preprocessing leakage.

This initial model uses default (unweighted) Logistic Regression; class weighting and other algorithms can be compared later.

In [ ]:
# ============================================================
# PROJECT 3 - STEP 4B: LOGISTIC REGRESSION PIPELINE
# ============================================================

# Import preprocessing tools.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

# Import the pipeline class.
from sklearn.pipeline import Pipeline

# Import Logistic Regression.
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# 1. IDENTIFY NUMERICAL AND CATEGORICAL PREDICTORS
# ------------------------------------------------------------

# Select numerical variables from the training dataset.
numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Select categorical predictors.
categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("FEATURE TYPES")
print("-" * 55)

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))


# ------------------------------------------------------------
# 2. CREATE THE PREPROCESSOR
# ------------------------------------------------------------

# StandardScaler:
# Standardizes numerical features using statistics
# calculated from the training dataset only.
#
# OneHotEncoder:
# Converts categorical variables into numerical columns.
#
# handle_unknown="ignore":
# Allows unseen validation categories without errors.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 3. CREATE THE LOGISTIC REGRESSION PIPELINE
# ------------------------------------------------------------

logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


# ------------------------------------------------------------
# 4. TRAIN THE MODEL
# ------------------------------------------------------------

# The pipeline learns preprocessing parameters
# and model coefficients from training data only.

logistic_model.fit(
    X_train,
    y_train
)

print("\nLogistic Regression training completed.")

## 4. Evaluate Logistic Regression on Validation Data

**Threshold-dependent metrics:** Accuracy, precision, recall, F1, and the confusion matrix use the classifier's default 0.50 threshold.

**Threshold-independent ranking metrics:** ROC-AUC and Average Precision (AP) use predicted churn probabilities. AP is particularly informative here because churn is rare.

**Interpretation:** High ROC-AUC does not guarantee high recall at a threshold of 0.50.

In [ ]:
# ============================================================
# PROJECT 3 - STEP 4C: LOGISTIC REGRESSION EVALUATION
# ============================================================

# Import classification metrics.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# 1. GENERATE VALIDATION PREDICTIONS
# ------------------------------------------------------------

# Predict customer churn classes.
logistic_predictions = logistic_model.predict(
    X_val
)

# Predict the probability of churn (class 1).
logistic_probabilities = (
    logistic_model.predict_proba(X_val)[:, 1]
)


# ------------------------------------------------------------
# 2. CALCULATE CLASSIFICATION METRICS
# ------------------------------------------------------------

logistic_accuracy = accuracy_score(
    y_val,
    logistic_predictions
)

logistic_precision = precision_score(
    y_val,
    logistic_predictions,
    zero_division=0
)

logistic_recall = recall_score(
    y_val,
    logistic_predictions,
    zero_division=0
)

logistic_f1 = f1_score(
    y_val,
    logistic_predictions,
    zero_division=0
)

# ROC-AUC evaluates how well the model ranks churners
# above non-churners across classification thresholds.
logistic_roc_auc = roc_auc_score(
    y_val,
    logistic_probabilities
)

# Average Precision summarizes the precision-recall curve.
# It is especially informative for rare positive outcomes.
logistic_ap = average_precision_score(
    y_val,
    logistic_probabilities
)


# ------------------------------------------------------------
# 3. DISPLAY THE RESULTS
# ------------------------------------------------------------

print("LOGISTIC REGRESSION VALIDATION RESULTS")
print("-" * 55)

print(f"Accuracy: {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall: {logistic_recall:.4f}")
print(f"F1-score: {logistic_f1:.4f}")
print(f"ROC-AUC: {logistic_roc_auc:.4f}")
print(f"Average Precision (AP): {logistic_ap:.4f}")


# ------------------------------------------------------------
# 4. DISPLAY THE CONFUSION MATRIX
# ------------------------------------------------------------

logistic_cm = confusion_matrix(
    y_val,
    logistic_predictions,
    labels=[0, 1]
)

print("\nCONFUSION MATRIX")
print("-" * 55)

print(logistic_cm)


# ------------------------------------------------------------
# 5. DISPLAY CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nCLASSIFICATION REPORT")
print("-" * 55)

print(
    classification_report(
        y_val,
        logistic_predictions,
        labels=[0, 1],
        target_names=["No Churn", "Churn"],
        zero_division=0
    )
)

## 5. Explore the Classification Threshold Trade-off

Lower thresholds flag more potential churners, typically increasing recall while also increasing false positives. Compare the **number of customers flagged** as well as precision and recall.

Do **not** finalize a threshold based only on the highest validation F1. A business operating rule should later account for outreach capacity, intervention costs, calibration, and whether a customer is expected to **benefit** from contact.

In [ ]:
# ============================================================
# PROJECT 3 - STEP 4D: CLASSIFICATION THRESHOLD ANALYSIS
# ============================================================

import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# ------------------------------------------------------------
# 1. DEFINE CLASSIFICATION THRESHOLDS
# ------------------------------------------------------------

# These thresholds will determine when a customer
# is classified as likely to churn.
thresholds = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50
]

# Create an empty list for storing evaluation results.
threshold_results = []

# ------------------------------------------------------------
# 2. EVALUATE EACH THRESHOLD
# ------------------------------------------------------------

for threshold in thresholds:

    # Convert predicted probabilities into binary predictions.
    # Customers with predicted probabilities greater than
    # or equal to the threshold are classified as churners.
    predictions = (
        logistic_probabilities >= threshold
    ).astype(int)

    # Calculate the classification metrics.
    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        predictions,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    # Count the number of customers predicted to churn.
    predicted_churners = predictions.sum()

    # Store the results for this threshold.
    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "Predicted_Churners": predicted_churners
    })

# ------------------------------------------------------------
# 3. CREATE A COMPARISON TABLE
# ------------------------------------------------------------

threshold_summary = pd.DataFrame(
    threshold_results
)

print("LOGISTIC REGRESSION THRESHOLD COMPARISON")
print("-" * 65)

display(threshold_summary.round(4))

## 6. Class-weighted Logistic Regression

`class_weight="balanced"` penalizes errors on rare churners more heavily. Compare at threshold 0.50 without changing the split.

In [ ]:
from sklearn.base import clone
weighted_logistic_model = clone(logistic_model)
weighted_logistic_model.set_params(classifier__class_weight="balanced")
weighted_logistic_model.fit(X_train, y_train)
weighted_probabilities = weighted_logistic_model.predict_proba(X_val)[:, 1]
weighted_predictions = (weighted_probabilities >= 0.50).astype(int)
print(pd.DataFrame([{ "Model":"Class-weighted Logistic", "Precision":precision_score(y_val,weighted_predictions,zero_division=0),"Recall":recall_score(y_val,weighted_predictions,zero_division=0),"F1":f1_score(y_val,weighted_predictions,zero_division=0),"ROC_AUC":roc_auc_score(y_val,weighted_probabilities),"AP":average_precision_score(y_val,weighted_probabilities)}]).round(4))
print(confusion_matrix(y_val,weighted_predictions,labels=[0,1]))

## 7. Compare logistic decision thresholds

Thresholds change how risk scores become positive classifications; they do not change the underlying ranking.

In [ ]:
thresholds = [0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50]
rows=[]
for name,scores in [("Logistic",logistic_probabilities),("Weighted Logistic",weighted_probabilities)]:
    for threshold in thresholds:
        pred=(scores>=threshold).astype(int)
        rows.append({"Model":name,"Threshold":threshold,"Precision":precision_score(y_val,pred,zero_division=0),"Recall":recall_score(y_val,pred,zero_division=0),"F1":f1_score(y_val,pred,zero_division=0),"Flagged":int(pred.sum())})
logistic_thresholds=pd.DataFrame(rows)
display(logistic_thresholds.round(4))

## 8. Random Forest baseline

The tree ensemble can model nonlinear relationships. Numerical features are passed through; categorical features are one-hot encoded.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_preprocessor=ColumnTransformer(transformers=[("numerical","passthrough",numerical_features),("categorical",OneHotEncoder(handle_unknown="ignore"),categorical_features)])
rf_model=Pipeline(steps=[("preprocessing",rf_preprocessor),("classifier",RandomForestClassifier(n_estimators=300,max_depth=10,min_samples_leaf=5,random_state=42,n_jobs=-1))])
rf_model.fit(X_train,y_train)
rf_probabilities=rf_model.predict_proba(X_val)[:,1]
rf_predictions=(rf_probabilities>=0.50).astype(int)
print("Random Forest ROC-AUC",round(roc_auc_score(y_val,rf_probabilities),4),"AP",round(average_precision_score(y_val,rf_probabilities),4))
print(confusion_matrix(y_val,rf_predictions,labels=[0,1]))

## 9. Random Forest thresholds

A default threshold of 0.50 may miss all churners. Test lower thresholds using validation data only.

In [ ]:
rf_thresholds=[0.02,0.03,0.05,0.07,0.10,0.15,0.20,0.30,0.40,0.50]
rf_rows=[]
for threshold in rf_thresholds:
    pred=(rf_probabilities>=threshold).astype(int)
    rf_rows.append({"Threshold":threshold,"Precision":precision_score(y_val,pred,zero_division=0),"Recall":recall_score(y_val,pred,zero_division=0),"F1":f1_score(y_val,pred,zero_division=0),"Flagged":int(pred.sum())})
display(pd.DataFrame(rf_rows).round(4))

## 10. Original CatBoost

CatBoost natively handles categorical predictors. This fixed configuration is compared against the Logistic Regression benchmark.

In [ ]:
from catboost import CatBoostClassifier
catboost_categorical=X_train.select_dtypes(include=["object","string","category"]).columns.tolist()
X_train_cb=X_train.copy();X_val_cb=X_val.copy()
for column in catboost_categorical:
    X_train_cb[column]=X_train_cb[column].astype(str)
    X_val_cb[column]=X_val_cb[column].astype(str)
catboost_model=CatBoostClassifier(iterations=500,depth=6,learning_rate=0.05,loss_function="Logloss",random_seed=42,verbose=False)
catboost_model.fit(X_train_cb,y_train,cat_features=catboost_categorical)
catboost_probabilities=catboost_model.predict_proba(X_val_cb)[:,1]
print("CatBoost ROC-AUC",round(roc_auc_score(y_val,catboost_probabilities),4),"AP",round(average_precision_score(y_val,catboost_probabilities),4))

## 11. CatBoost thresholds

Identify the exploratory validation precision/recall trade-off. Do not choose a final operating rule using the held-out test set.

In [ ]:
cb_thresholds=[0.02,0.03,0.05,0.07,0.10,0.15,0.20,0.25,0.30,0.40,0.50]
cb_rows=[]
for threshold in cb_thresholds:
    pred=(catboost_probabilities>=threshold).astype(int)
    cb_rows.append({"Threshold":threshold,"Precision":precision_score(y_val,pred,zero_division=0),"Recall":recall_score(y_val,pred,zero_division=0),"F1":f1_score(y_val,pred,zero_division=0),"Flagged":int(pred.sum())})
display(pd.DataFrame(cb_rows).round(4))

## 12. Precision–recall curves

Average Precision summarizes ranking quality across thresholds; the prevalence reference reflects the rare churn outcome.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
probability_models={"Logistic Regression":logistic_probabilities,"Weighted Logistic":weighted_probabilities,"Random Forest":rf_probabilities,"Original CatBoost":catboost_probabilities}
plt.figure(figsize=(9,6))
for name,scores in probability_models.items():
    precision,recall,_=precision_recall_curve(y_val,scores)
    plt.plot(recall,precision,label=f"{name} (AP={average_precision_score(y_val,scores):.4f})")
plt.axhline(y_val.mean(),ls="--",label=f"Prevalence ({y_val.mean():.4f})")
plt.xlabel("Recall");plt.ylabel("Precision");plt.title("Validation precision–recall curves")
plt.legend();plt.grid(alpha=0.2);plt.tight_layout();plt.show()

## 13. Training-only hyperparameter search

RandomizedSearchCV searches 12 parameter configurations across 3 stratified folds on the training partition. `cat_features` is supplied at fit-time to avoid a CatBoost/scikit-learn cloning issue. Cross-validation AP must not be directly compared with separate validation AP.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV,StratifiedKFold
catboost_search_model=CatBoostClassifier(loss_function="Logloss",random_seed=42,verbose=False,thread_count=2)
parameter_grid={"iterations":[300,500,700],"depth":[4,6,8],"learning_rate":[0.03,0.05,0.1],"l2_leaf_reg":[3,5,7]}
catboost_search=RandomizedSearchCV(catboost_search_model,parameter_grid,n_iter=12,scoring="average_precision",cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42),random_state=42,n_jobs=1,verbose=1,refit=True)
catboost_search.fit(X_train_cb,y_train,cat_features=catboost_categorical)
best_catboost_model=catboost_search.best_estimator_
print("Best parameters:",catboost_search.best_params_)
print("Training cross-validation AP:",round(catboost_search.best_score_,4))

## 14. Compare original and tuned models

Assess both on identical validation customers. A larger training-CV AP does not guarantee a larger validation AP.

In [ ]:
optimized_probabilities=best_catboost_model.predict_proba(X_val_cb)[:,1]
rows=[]
for name,scores in [("Original CatBoost",catboost_probabilities),("Tuned CatBoost",optimized_probabilities)]:
    pred=(scores>=0.05).astype(int)
    rows.append({"Model":name,"ROC_AUC":roc_auc_score(y_val,scores),"AP":average_precision_score(y_val,scores),"Precision_0.05":precision_score(y_val,pred,zero_division=0),"Recall_0.05":recall_score(y_val,pred,zero_division=0),"F1_0.05":f1_score(y_val,pred,zero_division=0),"Flagged_0.05":int(pred.sum())})
display(pd.DataFrame(rows).round(4))

## 15. Business contact capacity — Top-K

Rank the same validation customers at K=50,100,200,300,500. Risk ranking does **not** establish responsiveness to a retention offer.

In [ ]:
models={"Logistic":logistic_probabilities,"Weighted Logistic":weighted_probabilities,"Random Forest":rf_probabilities,"Original CatBoost":catboost_probabilities,"Tuned CatBoost":optimized_probabilities}
actual=np.asarray(y_val)
top_k=[]
for name,scores in models.items():
    ranked=np.argsort(-np.asarray(scores))
    for k in [50,100,200,300,500]:
        found=int(actual[ranked[:k]].sum())
        top_k.append({"Model":name,"K":k,"Churners":found,"Precision_at_K":found/k,"Recall_at_K":found/int(actual.sum())})
display(pd.DataFrame(top_k).round(4))

## 16. Probability calibration diagnostic

Brier score measures accuracy of probability forecasts; strong rankings may still be miscalibrated. Here curves are diagnostic only, not fitted probability-calibration models.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
calibration_models={"Logistic Regression":logistic_probabilities,"Original CatBoost":catboost_probabilities}
plt.figure(figsize=(8,5))
for name,scores in calibration_models.items():
    observed,forecast=calibration_curve(y_val,scores,n_bins=10,strategy="quantile")
    plt.plot(forecast,observed,marker="o",label=f"{name} (Brier={brier_score_loss(y_val,scores):.5f})")
plt.plot([0,0.30],[0,0.30],"--",label="Ideal")
plt.xlim(0,0.30);plt.ylim(0,0.30)
plt.xlabel("Mean predicted probability");plt.ylabel("Observed churn rate")
plt.title("Validation calibration (zoomed)");plt.legend();plt.grid(alpha=0.2);plt.tight_layout();plt.show()

## 17. Locked final evaluation — held-out test

**Predeclared choice:** original CatBoost (500 iterations, depth 6, learning rate 0.05), decision threshold **0.05**, and contact capacity **K=100**. Primary metric AP; report ROC-AUC, accuracy, precision, recall, F1, confusion matrix, and Precision@100/Recall@100. All prior choices used training/validation data only. **Do not retune using this test result.**

In [ ]:
from sklearn.metrics import classification_report
X_test_cb=X_test.copy()
for column in catboost_categorical:
    X_test_cb[column]=X_test_cb[column].astype(str)
test_probabilities=catboost_model.predict_proba(X_test_cb)[:,1]
test_predictions=(test_probabilities>=0.05).astype(int)
test_metrics={"Accuracy":accuracy_score(y_test,test_predictions),"Precision":precision_score(y_test,test_predictions,zero_division=0),"Recall":recall_score(y_test,test_predictions,zero_division=0),"F1":f1_score(y_test,test_predictions,zero_division=0),"ROC_AUC":roc_auc_score(y_test,test_probabilities),"AP":average_precision_score(y_test,test_probabilities)}
print("LOCKED TEST METRICS")
display(pd.DataFrame([test_metrics]).round(4))
print("Confusion matrix:\n",confusion_matrix(y_test,test_predictions,labels=[0,1]))
print(classification_report(y_test,test_predictions,zero_division=0))
ranked=np.argsort(-test_probabilities)
identified=int(np.asarray(y_test)[ranked[:100]].sum())
print("Top 100 churners identified:",identified,"Precision@100:",identified/100,"Recall@100:",identified/int(y_test.sum()))

## 18. Final findings and limitations

**Recorded held-out test results (original CatBoost, threshold 0.05):** accuracy **0.9307**, precision **0.2123**, recall **0.3827**, F1 **0.2731**, ROC-AUC **0.8437**, AP **0.1975**, confusion matrix `[[2184,115],[50,31]]`. Among the top 100 by predicted churn risk, **21** churned (Precision@100 = **0.21**; Recall@100 = **0.2593**). These are historical project results; rerunning the preceding cell independently reproduces or checks them.

**Interpretation:** The model provides useful risk ranking, but false positives and missed churners remain. The anonymized predictors limit interpretation. Churn-risk ranking does not show who would benefit from retention treatment; Notebook 3 addresses this exploratory question. Test performance comes from one reserved split and may vary across cohorts. Do not interpret predicted risk as proof of causation.